<a href="https://colab.research.google.com/github/victor-torres05/Assistente-de-Acessibilidade-por-Voz/blob/main/Assistente_de_Acessibilidade_por_Voz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ⚠️ PRÉ-REQUISITOS ANTES DE EXECUTAR
#
# 1. API Key da OpenAI:
#    - Crie uma conta em platform.openai.com
#    - Adicione crédito em platform.openai.com/settings/billing (mínimo $5)
#    - Gere sua chave em platform.openai.com/api-keys
#    - Cole sua chave na Célula 2 onde está indicado
#
# 2. Modelo utilizado: gpt-3.5-turbo
#    - Se sua conta tiver acesso ao gpt-4, pode substituir na Célula 6
#
# 3. Microfone:
#    - Permita o acesso ao microfone quando o navegador solicitar
#    - Você terá 5 segundos para fazer sua pergunta (ajustável na Célula 5)

**Célula 1 - Instalação**

In [1]:
!pip install git+https://github.com/openai/whisper.git -q
!pip install openai
!pip install gTTS
!pip install PyPDF2

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 11.6 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 17.5 MB/s eta 0:00:00


**Célula 2 — Imports e configuração**


In [13]:
import os
import openai
import whisper
from gtts import gTTS
from IPython.display import display, Audio
import PyPDF2

# Idioma fixo para respostas e síntese de voz
LANGUAGE = 'pt'

# Insira sua API Key aqui
os.environ['OPENAI_API_KEY'] = ''
openai.api_key = os.environ.get('OPENAI_API_KEY')

from openai import OpenAI
client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

# Carrega o modelo Whisper
model = whisper.load_model("small")
print("✅ Tudo carregado!")

✅ Tudo carregado!


**Célula 3 — Input do texto (texto livre ou PDF)**

In [3]:
# @title
input_type = input("Você vai inserir (1) texto colado ou (2) PDF? Digite 1 ou 2: ").strip()

if input_type == "1":
    print("Cole o texto abaixo e pressione Enter duas vezes quando terminar:\n")
    lines = []
    while True:
        line = input()
        if line == "":
            break
        lines.append(line)
    document_text = "\n".join(lines)

elif input_type == "2":
    from google.colab import files
    print("Faça o upload do PDF:")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]

    reader = PyPDF2.PdfReader(filename)
    document_text = ""
    for page in reader.pages:
        document_text += page.extract_text()

print(f"\n✅ Texto carregado ({len(document_text)} caracteres).")

Você vai inserir (1) texto colado ou (2) PDF? Digite 1 ou 2: 1
Cole o texto abaixo e pressione Enter duas vezes quando terminar:

A procrastinação — o ato de adiar tarefas — é uma das maiores armadilhas da produtividade moderna. Mesmo sabendo que deveríamos começar, acabamos empurrando as responsabilidades. Mas existem formas práticas e comprovadas de parar de procrastinar e começar a agir, tanto no trabalho quanto nos estudos.  Tratar uma questão praticamente intrínseca às pessoas (salvo exceções) com técnicas ditas “fáceis”, como hacks, então, tem uma eficiência limitada. Por isso, o site Quartz criou um guia, que combina hacks com táticas pessoais para você examinar seus processos internos. E, assim, criar uma abordagem própria para entender como parar de procrastinar de uma vez por todas.  Leia também: Gerenciamento de tempo e cronograma para o TCC: como evitar a procrastinação e cumprir prazos   Procrastinação: o que é e por que acontece O que é procrastinar? Procrastinar é poster

**Célula 4 — Inicializa o histórico da conversa**


In [14]:
# O system prompt instrui o GPT a usar somente o texto fornecido,
# responder sempre em português e de forma simples e acessível.
system_prompt = f"""Você é um assistente de acessibilidade.
Sua função é responder perguntas sobre o texto abaixo de forma clara, simples e acessível.
Sempre responda em português, independentemente do idioma da pergunta ou do texto.
Use linguagem simples, evite jargões técnicos e seja direto.
Se a pergunta não estiver relacionada ao texto, diga educadamente que só pode responder sobre ele.

TEXTO:
\"\"\"
{document_text}
\"\"\"
"""

conversation_history = [
    {"role": "system", "content": system_prompt}
]

print("✅ Assistente pronto! Execute a próxima célula para começar.")

✅ Assistente pronto! Execute a próxima célula para começar.


**Célula 5 — Gravação do áudio (JavaScript + Python)**


In [15]:
# Célula de gravação — execute esta célula a cada nova pergunta
from IPython.display import Javascript
from google.colab import output
import base64

RECORD_FILE = "/content/question_audio.wav"
DURATION = 5  # segundos de gravação — ajuste conforme necessário

record_js = f"""
async function recordAudio() {{
  const stream = await navigator.mediaDevices.getUserMedia({{ audio: true }});
  const recorder = new MediaRecorder(stream);
  const chunks = [];

  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();

  await new Promise(resolve => setTimeout(resolve, {DURATION * 1000}));

  recorder.stop();
  stream.getTracks().forEach(t => t.stop());

  await new Promise(resolve => setTimeout(resolve, 300));

  const blob = new Blob(chunks, {{ type: 'audio/wav' }});
  const reader = new FileReader();
  reader.readAsDataURL(blob);

  await new Promise(resolve => reader.onload = resolve);
  const base64Audio = reader.result.split(',')[1];

  google.colab.kernel.invokeFunction('save_audio', [base64Audio], {{}});
}}

recordAudio();
"""

def save_audio(base64_audio):
    audio_bytes = base64.b64decode(base64_audio)
    with open(RECORD_FILE, 'wb') as f:
        f.write(audio_bytes)
    print("✅ Áudio salvo! Execute a próxima célula para processar.")

output.register_callback('save_audio', save_audio)

print(f"🎙️ Ouvindo sua pergunta... ({DURATION} segundos)")
display(Javascript(record_js))

🎙️ Ouvindo sua pergunta... (5 segundos)


<IPython.core.display.Javascript object>

✅ Áudio salvo! Execute a próxima célula para processar.


**Célula 6 — Transcrição, resposta e síntese de voz**

In [17]:
RESPONSE_FILE = "/content/response_audio.mp3"

# 1. Transcreve com Whisper
print("🔄 Transcrevendo...")
result = model.transcribe(RECORD_FILE, fp16=False)
question = result["text"]
print(f"📝 Você perguntou: {question}")

# Detecta encerramento
if "finalizar assistente" in question.lower():
    print("👋 Encerrando o assistente. Até mais!")
else:
    # 2. Adiciona a pergunta ao histórico e envia ao GPT
    conversation_history.append({"role": "user", "content": question})

    print("🤖 Consultando o assistente...")
    response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=conversation_history
)
answer = response.choices[0].message.content
conversation_history.append({"role": "assistant", "content": answer})

print(f"\n💬 Resposta: {answer}\n")

# 3. Converte resposta em voz
tts = gTTS(text=answer, lang=LANGUAGE, slow=False)
tts.save(RESPONSE_FILE)
display(Audio(RESPONSE_FILE, autoplay=True))

print("\n🔁 Execute a Célula 5 novamente para fazer outra pergunta.")

🔄 Transcrevendo...
📝 Você perguntou:  Qual é a melhor forma de parar de procrastinar?
🤖 Consultando o assistente...


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}